# खर्च दाबी विश्लेषण

यो नोटबुकले कसरी एजेन्टहरू सिर्जना गर्ने देखाउँछ जुन प्लगइनहरू प्रयोग गरेर स्थानीय रसिद छविहरूबाट यात्रा खर्च प्रशोधन गर्छन्, खर्च दाबी इमेल उत्पन्न गर्छन्, र खर्च डेटा प्याई चार्ट प्रयोग गरेर दृश्य बनाउँछ। एजेन्टहरूले कार्य सन्दर्भको आधारमा गतिशील रूपमा कार्यहरू चयन गर्छन्।

कदमहरू:
1. OCR एजेन्टले स्थानीय रसिद छवि प्रशोधन गरी यात्रा खर्च डेटा निकाल्छ।
2. इमेल एजेन्टले खर्च दाबी इमेल उत्पन्न गर्छ।

### यात्रा खर्च अवस्थाको उदाहरण:
कल्पना गर्नुहोस् तपाईं अर्को शहरमा व्यवसायिक बैठकका लागि यात्रा गर्दै हुनुहुन्छ। तपाईंको कम्पनीले सबै उचित यात्रा सम्बन्धित खर्चहरू फिर्ता दिने नीति राखेको छ। यहाँ सम्भावित यात्रा खर्चहरूको विवरण छ:
- यातायात:
तपाईंको गृह शहरदेखि गन्तव्य शहरसम्मको राउन्ड ट्रिपको लागि हवाई टिकट।
विमानस्थल जाने र फर्कने ट्याक्सी वा राइड-हेलिंग सेवा।
गन्तव्य शहरमा स्थानीय यातायात (जस्तै सार्वजनिक यातायात, भाडामा कार, वा ट्याक्सी)।

- आवास:
बैठक स्थल नजिकैको मध्यम स्तरको व्यवसायिक होटलमा तीन रातको बसाइ।

- भोजन:
कम्पनीको प्रति diem नीतिअनुसार बिहानको खाजा, दिउँसोको खाना, र रातको खानाको दैनिक भत्ता।

- विविध खर्चहरू:
विमानस्थलमा पार्किङ शुल्क।
होटलमा इन्टरनेट पहुँच शुल्क।
टिप्स वा साना सेवा शुल्कहरू।

- कागजातहरू:
तपाईंले सबै रसिदहरू (उडान, ट्याक्सी, होटल, खाना, आदि) र फिर्ताका लागि पूरा गरिएको खर्च प्रतिवेदन पेश गर्नुहुन्छ।


## आवश्यक पुस्तकालयहरू आयात गर्नुहोस्

नोटबुकका लागि आवश्यक पुस्तकालयहरू र मोड्युलहरू आयात गर्नुहोस्।


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## खर्च मोडेलहरू परिभाषित गर्नुहोस्

व्यक्तिगत खर्चहरूका लागि एक Pydantic मोडेल बनाउनुहोस् र प्रयोगकर्ताको सोधलाई संरचित खर्च डाटामा रूपान्तरण गर्न ExpenseFormatter नामक क्लास बनाउनुहोस्।

प्रत्येक खर्च यस ढाँचामा प्रतिनिधित्व गरिनेछ:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## उपकरण परिभाषित गर्दै - इमेल निर्माण

खर्च दावी पेश गर्नको लागि इमेल निर्माण गर्ने उपकरण फंक्शन बनाउनुहोस्।
- यो उपकरण Microsoft Agent Framework बाट `@tool` डेकोरेटर प्रयोग गर्छ।
- यो खर्चहरूको कुल रकम गणना गर्छ र विवरणहरू इमेल को शरीरमा फर्म्याट गर्छ।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# रसिद छविहरूबाट यात्रा खर्च निकाल्नको लागि उपकरण

रसिद छविहरूबाट यात्रा खर्च निकाल्नको लागि उपकरण कार्य सिर्जना गर्नुहोस्।
- यो उपकरण Microsoft Agent Framework बाट `@tool` डेकोरेटर प्रयोग गर्दछ।
- यसले रसिद छवि पढ्छ, यसलाई base64 मा इन्कोड गर्दछ, र एजेन्टलाई विश्लेषणका लागि डेटा URI फिर्ता गर्छ।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्च प्रशोधन

एजेण्टहरू परिभाषित गर्नुहोस् र तिनीहरूलाई `WorkflowBuilder` प्रयोग गरी एक क्रमिक कामको फलोमा जडान गर्नुहोस्।  
- OCR एजेण्टले `load_receipt_image` उपकरण प्रयोग गरी रसिद छविबाट संरचित खर्च डेटा निकाल्छ।  
- इमेल एजेण्टले निकालिएको डेटा लिएर `generate_expense_email` उपकरण प्रयोग गरी व्यावसायिक खर्च दावी इमेल निर्माण गर्छ।  
- `WorkflowBuilder` मा `add_edge` प्रयोग गरी एक क्रमिक पाइपलाइन बनाउँछ: OCR एजेण्ट → इमेल एजेण्ट।


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## मुख्य कार्य

अनुसूची workflow निर्माण गर्नुहोस् र रिसिप्ट छवि प्रशोधन गर्न र खर्च दाबी ईमेल सिर्जना गर्न यसलाई चलाउनुहोस्।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही अनुवादको प्रयास गर्छौँ, तर कृपया जानकार हुनुहोस् कि स्वचालित अनुवादमा त्रुटि वा असत्यता हुन सक्छ। मूल दस्तावेज़ यसको मूल भाषामा नै प्रमाणिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
